**Data Ingestion**

In [0]:
path = "/Volumes/rg_azure_databricks/default/filestore"
patients_df = spark.read.option("header",True).option("inferSchema",True).csv(f"{path}/patients.csv")
display(patients_df)

patient_id,patient_name,city,state,age,gender,insurance_status
P101,Rahul Sharma,Hyderabad,Telangana,35,Male,Active
P102,Priya Reddy,Bangalore,Karnataka,29,Female,Active
P103,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive
P104,Sneha Patel,Delhi,Delhi,31,Female,Active
P105,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active
P106,Neha Singh,Pune,Maharashtra,38,Female,Inactive
P107,Arjun Verma,Hyderabad,Telangana,26,Male,Active
P108,Meera Nair,Kochi,Kerala,48,Female,Active


In [0]:
doctors_df = spark.read.option("header",True).option("inferSchema",True).csv(f"{path}/doctors.csv")
display(doctors_df)

doctor_id,doctor_name,department,city,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,1500
D102,Dr. Priya,Neurology,Bangalore,2000
D103,Dr. Anita,Dermatology,Chennai,1000
D104,Dr. Suresh,Orthopedics,Mumbai,2500
D105,Dr. Meera,Pediatrics,Delhi,1200
D106,Dr. Kiran,Cardiology,Hyderabad,3000


In [0]:
appointments_df = spark.read.option("header",True).option("inferSchema",True).csv(f"{path}/appointments.csv")
display(appointments_df)

appointment_id,patient_id,doctor_id,appointment_date,diagnosis,bill_amount,status
A1001,P101,D101,2026-06-01,Heart Checkup,5000,Completed
A1002,P102,D102,2026-06-01,Migraine,3500,Completed
A1003,P103,D103,2026-06-02,Skin Allergy,2000,Pending
A1004,P104,D104,2026-06-02,Fracture,12000,Completed
A1005,P105,D105,2026-06-03,Fever,1500,Completed
A1006,P106,D106,2026-06-03,Heart Checkup,7000,Completed
A1007,P107,D101,2026-06-04,Chest Pain,5500,Completed
A1008,P108,D103,2026-06-04,Skin Infection,2500,Pending
A1009,P101,D106,2026-06-05,Cardiac Review,6500,Completed
A1010,P104,D104,2026-06-05,Back Pain,4500,Cancelled


In [0]:
preferences_df = spark.read.option("multiline",True).json(f"{path}/patient_preferences.json")
display(preferences_df)

contact,patient_id,preferred_hospital
"List(rahul@mail.com, 9876500011)",P101,Apollo Hospital
"List(priya@mail.com, 9876500012)",P102,Yashoda Hospital
"List(sneha@mail.com, 9876500014)",P104,Care Hospital
"List(meera@mail.com, 9876500018)",P108,Apollo Hospital


In [0]:
patients_df.printSchema()
doctors_df.printSchema()
appointments_df.printSchema()
preferences_df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- insurance_status: string (nullable = true)

root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- city: string (nullable = true)
 |-- consultation_fee: integer (nullable = true)

root
 |-- appointment_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- appointment_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- status: string (nullable = true)

root
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- preferred_hospita

In [0]:
patients_df.write.format("delta").mode("overwrite").saveAsTable("bronze_patients")
doctors_df.write.format("delta").mode("overwrite").saveAsTable("bronze_doctors")
appointments_df.write.format("delta").mode("overwrite").saveAsTable("bronze_appointments")
preferences_df.write.format("delta").mode("overwrite").saveAsTable("bronze_preferences")

**Data Cleaning and Transformation**

In [0]:
patients_df = patients_df.na.fill("Unknown")
doctors_df = doctors_df.na.fill("Unknown")
appointments_df = appointments_df.na.fill(0)

In [0]:
from pyspark.sql.functions import col

pref_flat = preferences_df.select(
    "patient_id",
    "preferred_hospital",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)
display(pref_flat)

patient_id,preferred_hospital,phone,email
P101,Apollo Hospital,9876500011,rahul@mail.com
P102,Yashoda Hospital,9876500012,priya@mail.com
P104,Care Hospital,9876500014,sneha@mail.com
P108,Apollo Hospital,9876500018,meera@mail.com


In [0]:
patients_df = patients_df.withColumnRenamed("city", "patient_city")
patient_pref_df = patients_df.join(
    pref_flat,
    "patient_id",
    "left"
)
display(patient_pref_df)

patient_id,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email
P101,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P102,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com
P103,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null
P104,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com
P105,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null
P106,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null
P107,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null
P108,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com


In [0]:
appt_patient_df = appointments_df.join(
    patient_pref_df,
    "patient_id",
    "left"
)
display(appt_patient_df)

patient_id,appointment_id,doctor_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email
P101,A1001,D101,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P102,A1002,D102,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com
P103,A1003,D103,2026-06-02,Skin Allergy,2000,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null
P104,A1004,D104,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com
P105,A1005,D105,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null
P106,A1006,D106,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null
P107,A1007,D101,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null
P108,A1008,D103,2026-06-04,Skin Infection,2500,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com
P101,A1009,D106,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com
P104,A1010,D104,2026-06-05,Back Pain,4500,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com


In [0]:
doctors_df = doctors_df.withColumnRenamed("city", "doctor_city")
final_df = appt_patient_df.join(
    doctors_df,
    "doctor_id",
    "left"
)
display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee
D101,P101,A1001,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500
D102,P102,A1002,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000
D103,P103,A1003,2026-06-02,Skin Allergy,2000,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000
D104,P104,A1004,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500
D105,P105,A1005,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200
D106,P106,A1006,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000
D101,P107,A1007,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500
D103,P108,A1008,2026-06-04,Skin Infection,2500,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000
D106,P101,A1009,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000
D104,P104,A1010,2026-06-05,Back Pain,4500,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500


In [0]:
from pyspark.sql.functions import *

final_df = final_df.withColumn(
    "final_bill",
    col("bill_amount") + col("consultation_fee")
)
display(final_df)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500,6500,2026-06,Adult
D102,P102,A1002,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000,5500,2026-06,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000,3000,2026-06,Adult
D104,P104,A1004,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,14500,2026-06,Adult
D105,P105,A1005,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200,2700,2026-06,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000,10000,2026-06,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500,7000,2026-06,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000,3500,2026-06,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000,9500,2026-06,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,7000,2026-06,Adult


In [0]:
final_df = final_df.withColumn(
    "appointment_month",
    date_format("appointment_date","yyyy-MM")
)
app = final_df.select("appointment_id","appointment_date","appointment_month")
display(app)

appointment_id,appointment_date,appointment_month
A1001,2026-06-01,2026-06
A1002,2026-06-01,2026-06
A1003,2026-06-02,2026-06
A1004,2026-06-02,2026-06
A1005,2026-06-03,2026-06
A1006,2026-06-03,2026-06
A1007,2026-06-04,2026-06
A1008,2026-06-04,2026-06
A1009,2026-06-05,2026-06
A1010,2026-06-05,2026-06


In [0]:
final_df = final_df.withColumn(
    "patient_age_group",
    when(col("age") >= 50,"Senior")
    .when(col("age") >= 30,"Adult")
    .otherwise("Young")
)
pat = final_df.select("patient_id","patient_name","age","patient_age_group")
display(pat)

patient_id,patient_name,age,patient_age_group
P101,Rahul Sharma,35,Adult
P102,Priya Reddy,29,Young
P103,Amit Kumar,42,Adult
P104,Sneha Patel,31,Adult
P105,Farhan Ali,55,Senior
P106,Neha Singh,38,Adult
P107,Arjun Verma,26,Young
P108,Meera Nair,48,Adult
P101,Rahul Sharma,35,Adult
P104,Sneha Patel,31,Adult


In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("silver_healthcare")

**Spark SQL**

In [0]:
final_df.createOrReplaceTempView("healthcare")

In [0]:
%sql
SELECT SUM(final_bill) AS total_revenue FROM healthcare;

total_revenue
69200


In [0]:
%sql
SELECT department,SUM(final_bill) AS revenue FROM healthcare
GROUP BY department
ORDER BY revenue DESC;

department,revenue
Cardiology,33000
Orthopedics,21500
Dermatology,6500
Neurology,5500
Pediatrics,2700


In [0]:
%sql
SELECT patient_city,SUM(final_bill) AS revenue FROM healthcare
GROUP BY patient_city
ORDER BY revenue DESC;

patient_city,revenue
Hyderabad,23000
Delhi,21500
Pune,10000
Bangalore,5500
Kochi,3500
Mumbai,3000
Chennai,2700


In [0]:
%sql
SELECT * FROM healthcare
WHERE status='Completed';

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500,6500,2026-06,Adult
D102,P102,A1002,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000,5500,2026-06,Young
D104,P104,A1004,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,14500,2026-06,Adult
D105,P105,A1005,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200,2700,2026-06,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000,10000,2026-06,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500,7000,2026-06,Young
D106,P101,A1009,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000,9500,2026-06,Adult


In [0]:
%sql
SELECT patient_name,SUM(final_bill) AS total_bill FROM healthcare
GROUP BY patient_name
ORDER BY total_bill DESC;

patient_name,total_bill
Sneha Patel,21500
Rahul Sharma,16000
Neha Singh,10000
Arjun Verma,7000
Priya Reddy,5500
Meera Nair,3500
Amit Kumar,3000
Farhan Ali,2700


**Window Functions**

In [0]:
from pyspark.sql.window import Window

doctor_window = Window.orderBy(desc("revenue"))

doctor_rank = final_df.groupBy(
    "doctor_id",
    "doctor_name"
).agg(
    sum("final_bill").alias("revenue")
).withColumn(
    "rank",
    rank().over(doctor_window)
)
display(doctor_rank)

doctor_id,doctor_name,revenue,rank
D104,Dr. Suresh,21500,1
D106,Dr. Kiran,19500,2
D101,Dr. Ramesh,13500,3
D103,Dr. Anita,6500,4
D102,Dr. Priya,5500,5
D105,Dr. Meera,2700,6


In [0]:
dept_window = Window.orderBy(desc("revenue"))

dept_rank = final_df.groupBy(
    "department"
).agg(
    sum("final_bill").alias("revenue")
).withColumn(
    "rank",
    rank().over(dept_window)
)
display(dept_rank)

department,revenue,rank
Cardiology,33000,1
Orthopedics,21500,2
Dermatology,6500,3
Neurology,5500,4
Pediatrics,2700,5


In [0]:
patient_window = Window.orderBy(desc("total_bill"))

top_patients = final_df.groupBy(
    "patient_id",
    "patient_name"
).agg(
    sum("final_bill").alias("total_bill")
).withColumn(
    "rank",
    rank().over(patient_window)
).filter("rank <= 3")
display(top_patients)

patient_id,patient_name,total_bill,rank
P104,Sneha Patel,21500,1
P101,Rahul Sharma,16000,2
P106,Neha Singh,10000,3


In [0]:
dept_doc_window = Window.partitionBy("department").orderBy(desc("revenue"))

top_doctors = final_df.groupBy(
    "department",
    "doctor_name"
).agg(
    sum("final_bill").alias("revenue")
).withColumn(
    "rank",
    rank().over(dept_doc_window)
).filter("rank=1")
display(top_doctors)

department,doctor_name,revenue,rank
Cardiology,Dr. Kiran,19500,1
Dermatology,Dr. Anita,6500,1
Neurology,Dr. Priya,5500,1
Orthopedics,Dr. Suresh,21500,1
Pediatrics,Dr. Meera,2700,1


In [0]:
run_window = Window.orderBy("appointment_date")

running_revenue = final_df.withColumn(
    "running_revenue",
    sum("final_bill").over(run_window)
)
display(running_revenue)

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group,running_revenue
D101,P101,A1001,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500,6500,2026-06,Adult,12000
D102,P102,A1002,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000,5500,2026-06,Young,12000
D103,P103,A1003,2026-06-02,Skin Allergy,2000,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000,3000,2026-06,Adult,29500
D104,P104,A1004,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,14500,2026-06,Adult,29500
D105,P105,A1005,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200,2700,2026-06,Senior,42200
D106,P106,A1006,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000,10000,2026-06,Adult,42200
D101,P107,A1007,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500,7000,2026-06,Young,52700
D103,P108,A1008,2026-06-04,Skin Infection,2500,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000,3500,2026-06,Adult,52700
D106,P101,A1009,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000,9500,2026-06,Adult,69200
D104,P104,A1010,2026-06-05,Back Pain,4500,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,7000,2026-06,Adult,69200


**Delta Lake**

In [0]:
final_df.write.format("delta").mode("overwrite").save("/delta/healthcare")

In [0]:
final_df.write.format("delta").mode("overwrite").saveAsTable("healthcare_delta")

In [0]:
%sql
CREATE TABLE healthcare_sql
USING DELTA
AS
SELECT * FROM healthcare;

num_affected_rows,num_inserted_rows


In [0]:
%sql 
DESCRIBE HISTORY healthcare_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-06-22T05:15:36Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(454155129810462),null,0622-033954-ly4ykbq3,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 10, numOutputBytes -> 8563)",null,Databricks-Runtime/18.x-cpu-ml-scala2.13


In [0]:
%sql
SELECT * FROM healthcare_delta VERSION AS OF 0;

doctor_id,patient_id,appointment_id,appointment_date,diagnosis,bill_amount,status,patient_name,patient_city,state,age,gender,insurance_status,preferred_hospital,phone,email,doctor_name,department,doctor_city,consultation_fee,final_bill,appointment_month,patient_age_group
D101,P101,A1001,2026-06-01,Heart Checkup,5000,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Ramesh,Cardiology,Hyderabad,1500,6500,2026-06,Adult
D102,P102,A1002,2026-06-01,Migraine,3500,Completed,Priya Reddy,Bangalore,Karnataka,29,Female,Active,Yashoda Hospital,9876500012,priya@mail.com,Dr. Priya,Neurology,Bangalore,2000,5500,2026-06,Young
D103,P103,A1003,2026-06-02,Skin Allergy,2000,Pending,Amit Kumar,Mumbai,Maharashtra,42,Male,Inactive,null,null,null,Dr. Anita,Dermatology,Chennai,1000,3000,2026-06,Adult
D104,P104,A1004,2026-06-02,Fracture,12000,Completed,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,14500,2026-06,Adult
D105,P105,A1005,2026-06-03,Fever,1500,Completed,Farhan Ali,Chennai,Tamil Nadu,55,Male,Active,null,null,null,Dr. Meera,Pediatrics,Delhi,1200,2700,2026-06,Senior
D106,P106,A1006,2026-06-03,Heart Checkup,7000,Completed,Neha Singh,Pune,Maharashtra,38,Female,Inactive,null,null,null,Dr. Kiran,Cardiology,Hyderabad,3000,10000,2026-06,Adult
D101,P107,A1007,2026-06-04,Chest Pain,5500,Completed,Arjun Verma,Hyderabad,Telangana,26,Male,Active,null,null,null,Dr. Ramesh,Cardiology,Hyderabad,1500,7000,2026-06,Young
D103,P108,A1008,2026-06-04,Skin Infection,2500,Pending,Meera Nair,Kochi,Kerala,48,Female,Active,Apollo Hospital,9876500018,meera@mail.com,Dr. Anita,Dermatology,Chennai,1000,3500,2026-06,Adult
D106,P101,A1009,2026-06-05,Cardiac Review,6500,Completed,Rahul Sharma,Hyderabad,Telangana,35,Male,Active,Apollo Hospital,9876500011,rahul@mail.com,Dr. Kiran,Cardiology,Hyderabad,3000,9500,2026-06,Adult
D104,P104,A1010,2026-06-05,Back Pain,4500,Cancelled,Sneha Patel,Delhi,Delhi,31,Female,Active,Care Hospital,9876500014,sneha@mail.com,Dr. Suresh,Orthopedics,Mumbai,2500,7000,2026-06,Adult


In [0]:
from pyspark.sql import Row

updates_data = [
    Row(patient_id="P101", patient_city="Noida", insurance_status="Inactive"),
    Row(patient_id="P103", patient_city="Bengaluru", insurance_status="Active"),
    Row(patient_id="P105", patient_city="Hyderabad", insurance_status="Active")
]

patient_updates = spark.createDataFrame(updates_data)
display(patient_updates)

patient_id,patient_city,insurance_status
P101,Noida,Inactive
P103,Bengaluru,Active
P105,Hyderabad,Active


In [0]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forName(spark, "healthcare_delta")

deltaTable.alias("t").merge(
    patient_updates.alias("s"),
    "t.patient_id = s.patient_id"
).whenMatchedUpdate(
    set={
        "patient_city": "s.patient_city",
        "insurance_status": "s.insurance_status"
    }
).execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
DESCRIBE HISTORY healthcare_delta;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-06-22T05:26:53Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(454155129810462),null,0622-033954-ly4ykbq3,3,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 30037, p25FileSize -> 8606, numDeletionVectorsRemoved -> 1, minFileSize -> 8606, numAddedFiles -> 1, maxFileSize -> 8606, p75FileSize -> 8606, p50FileSize -> 8606, numAddedBytes -> 8606)",null,Databricks-Runtime/18.x-cpu-ml-scala2.13
3,2026-06-22T05:26:50Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(patient_id#12183 = patient_id#12149)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(454155129810462),null,0622-033954-ly4ykbq3,2,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 21430, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 3345, materializeSourceTimeMs -> 288, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1255, numTargetRowsUpdated -> 4, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1782)",null,Databricks-Runtime/18.x-cpu-ml-scala2.13
2,2026-06-22T05:26:33Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(454155129810462),null,0622-033954-ly4ykbq3,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 4, numRemovedBytes -> 29991, p25FileSize -> 8607, numDeletionVectorsRemoved -> 1, minFileSize -> 8607, numAddedFiles -> 1, maxFileSize -> 8607, p75FileSize -> 8607, p50FileSize -> 8607, numAddedBytes -> 8607)",null,Databricks-Runtime/18.x-cpu-ml-scala2.13
1,2026-06-22T05:26:29Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,MERGE,"Map(predicate -> [""(patient_id#10447 = patient_id#10413)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [])",null,List(454155129810462),null,0622-033954-ly4ykbq3,0,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 3, numTargetBytesAdded -> 21428, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 4, executionTimeMs -> 3432, materializeSourceTimeMs -> 302, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1064, numTargetRowsUpdated -> 4, numOutputRows -> 4, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2023)",null,Databricks-Runtime/18.x-cpu-ml-scala2.13
0,2026-06-22T05:15:36Z,149040834249995,azuser7219_mml.local@karthikirisoutlook.onmicrosoft.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(454155129810462),null,0622-033954-ly4ykbq3,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles

In [0]:
%sql
OPTIMIZE healthcare_delta;

path,metrics
abfss://unity-catalog-storage@dbstorageoa7kihwyi26rk.dfs.core.windows.net/7405619834863317/__unitystorage/catalogs/87069ed1-8e5c-4c00-af7e-233acc1a57e9/tables/ad618fe1-1316-4d0d-bc4c-dc7d3e94840f,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, null, null, 0, 0, 1, 1, true, 0, 0, 1782106075940, 1782106078117, 4, 0, null, List(0, 0), null, 23, 23, 0, 0, null, null)"


In [0]:
%sql
OPTIMIZE healthcare_delta
ZORDER BY (patient_id);

path,metrics
abfss://unity-catalog-storage@dbstorageoa7kihwyi26rk.dfs.core.windows.net/7405619834863317/__unitystorage/catalogs/87069ed1-8e5c-4c00-af7e-233acc1a57e9/tables/ad618fe1-1316-4d0d-bc4c-dc7d3e94840f,"List(0, 0, List(null, null, 0.0, 0, 0), List(null, null, 0.0, 0, 0), 0, List(minCubeSize(107374182400), List(0, 0), List(1, 8606), 0, List(0, 0), 0, null), null, 0, 0, 1, 1, false, 0, 0, 1782106118769, 1782106120172, 4, 0, null, List(0, 0), null, 23, 23, 0, 0, null, null)"


In [0]:
%sql
VACUUM healthcare_delta RETAIN 168 HOURS;

path
abfss://unity-catalog-storage@dbstorageoa7kihwyi26rk.dfs.core.windows.net/7405619834863317/__unitystorage/catalogs/87069ed1-8e5c-4c00-af7e-233acc1a57e9/tables/ad618fe1-1316-4d0d-bc4c-dc7d3e94840f


**Visualization**

In [0]:
display(
    final_df.groupBy("department")
    .agg(sum("final_bill").alias("revenue"))
)

department,revenue
Neurology,5500
Dermatology,6500
Cardiology,33000
Pediatrics,2700
Orthopedics,21500


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    final_df.groupBy("patient_city")
    .agg(sum("final_bill").alias("revenue"))
)

patient_city,revenue
Bangalore,5500
Kochi,3500
Chennai,2700
Mumbai,3000
Pune,10000
Delhi,21500
Hyderabad,23000


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    final_df.groupBy("status").count()
)

status,count
Completed,7
Cancelled,1
Pending,2


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    doctor_rank
)

doctor_id,doctor_name,revenue,rank
D104,Dr. Suresh,21500,1
D106,Dr. Kiran,19500,2
D101,Dr. Ramesh,13500,3
D103,Dr. Anita,6500,4
D102,Dr. Priya,5500,5
D105,Dr. Meera,2700,6


Databricks visualization. Run in Databricks to view.

In [0]:
display(
    final_df.groupBy("appointment_date")
    .agg(sum("final_bill").alias("revenue"))
)

appointment_date,revenue
2026-06-03,12700
2026-06-05,16500
2026-06-02,17500
2026-06-01,12000
2026-06-04,10500


Databricks visualization. Run in Databricks to view.

**Tables & Views**

In [0]:
%sql
CREATE TABLE managed_healthcare
USING DELTA
AS SELECT * FROM healthcare;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TABLE external_healthcare
USING DELTA
LOCATION 'abfss://unity-catalog-storage@dbstorageoa7kihwyi26rk.dfs.core.windows.net/7405619834863317/external_healthcare'
AS
SELECT * FROM healthcare;

num_affected_rows,num_inserted_rows


In [0]:
%sql
CREATE TEMP VIEW temp_healthcare AS
SELECT * FROM healthcare;

In [0]:
%sql
CREATE GLOBAL TEMP VIEW global_healthcare AS
SELECT * FROM healthcare;

In [0]:
data = [
    ("Managed Table", "Available to all users with permissions in the metastore", "Exists until dropped", "Databricks manages both metadata and data"),
    ("External Table", "Available to all users with permissions", "Exists until dropped; data remains even if table is dropped", "Data stored at a user-specified external location"),
    ("Temporary View", "Current notebook/session only", "Removed automatically when session ends", "Metadata only; no data stored"),
    ("Global Temporary View", "Accessible across notebooks in same cluster using global_temp", "Exists until cluster terminates", "Metadata only; no data stored"),
]

df = spark.createDataFrame(data, ["Object", "Scope", "Lifetime", "Storage"])
display(df)

Object,Scope,Lifetime,Storage
Managed Table,Available to all users with permissions in the metastore,Exists until dropped,Databricks manages both metadata and data
External Table,Available to all users with permissions,Exists until dropped; data remains even if table is dropped,Data stored at a user-specified external location
Temporary View,Current notebook/session only,Removed automatically when session ends,Metadata only; no data stored
Global Temporary View,Accessible across notebooks in same cluster using global_temp,Exists until cluster terminates,Metadata only; no data stored
